# 07 · A tiny character-level language model

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilesh-salpe/visual-ai/blob/main/build-it/nn-fundamentals/07_tiny_char_lm.ipynb) &nbsp; 📖 Concept deck: [From MLP to language model](https://nilesh-salpe.github.io/visual-ai/nn-fundamentals/10-mlp-to-language-model/) &nbsp; ⏱️ Suggested time: **40 min**

**You will build:** a counting bigram model, then a Bengio-style neural model (embeddings → hidden layer → softmax over the next character) trained by cross-entropy, compared on held-out loss, and sampled.

**How to use this notebook**
1. Run the **Setup** cell.
2. For each task, fill in the `TODO` in the code cell, then run the **Tests** cell under it. You'll see ⏳ (not done yet), ✅ (passed) or ❌ (failed, with the reason).
3. Stuck? The **Reference solutions** at the bottom are collapsed. Try first.

Everything runs on CPU with NumPy, so no GPU is needed.

In [ ]:
# @title Setup: run this first
import numpy as np, math, time, itertools, collections
np.set_printoptions(precision=4, suppress=True)

def check(name, test, impl):
    """Run `test(impl)` and report: ⏳ not implemented, ✅ passed, ❌ failed with the reason."""
    try:
        test(impl)
        print(f"✅ {name}: all tests passed")
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet. Fill in the TODO in the cell above, then re-run this cell.")
    except AssertionError as e:
        print(f"❌ {name}: {e}")


TEXT = ("the cat sat on the mat. the dog sat on the log. a cat and a dog met on a mat. "
        "the cat saw the dog and the dog saw the cat. on the log sat a frog. the frog sat still. ") * 6
chars = sorted(set(TEXT)); V = len(chars); stoi = {c: i for i, c in enumerate(chars)}
data = np.array([stoi[c] for c in TEXT]); split = int(0.9 * len(data))
train, val = data[:split], data[split:]
def softmax(z):
    z = z - z.max(-1, keepdims=True); e = np.exp(z); return e / e.sum(-1, keepdims=True)

## Task 1: Counting bigram model

Count how often each character follows each other character in `train`, add `alpha` to every count (smoothing), normalise rows into probabilities, and return the mean negative log-likelihood (nats) of the next character on `val`.

In [ ]:
def bigram_loss(alpha=0.1):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 1: run after implementing
def _test_bigram_loss(f):
    L = f()
    assert L < np.log(V), f"must beat uniform ({np.log(V):.2f} nats), got {L:.2f}"
    assert f(alpha=100) > L, "heavy smoothing pushes the model toward uniform"

check("bigram_loss", _test_bigram_loss, bigram_loss)

## Task 2: A neural language model with a context of 3

Model (Bengio et al., 2003): embed each of the previous `ctx` characters with a shared table E (V × d), concatenate, $h = \tanh(x W_1 + b_1)$, logits $= h W_2 + b_2$, softmax over V.
Train with full-batch gradient descent on cross-entropy for `steps` steps. Return `(train_losses, val_loss)`. The gradient of softmax + cross-entropy w.r.t. logits is $(p - \text{onehot})/n$. For E, scatter-add the gradient of each embedding slot back into its row (`np.add.at`).

In [ ]:
def mlp_lm(ctx=3, d=8, hidden=64, lr=0.5, steps=1500, seed=0):
    r = np.random.default_rng(seed)
    def windows(seq):
        X = np.stack([seq[i:len(seq) - ctx + i] for i in range(ctx)], 1); return X, seq[ctx:]
    Xtr, Ytr = windows(train); Xva, Yva = windows(val)
    # TODO: init E, W1, b1, W2, b2; train; evaluate on Xva
    raise NotImplementedError

In [ ]:
# @title Tests for Task 2: run after implementing
def _test_mlp_lm(f):
    tl, vl = f()
    assert tl[-1] < tl[0] * 0.6, "training loss should drop substantially"
    C = np.full((V, V), 0.1); np.add.at(C, (train[:-1], train[1:]), 1); P = C / C.sum(1, keepdims=True)
    bg = -np.mean(np.log(P[val[:-1], val[1:]]))
    assert vl < bg - 0.2, f"3 characters of context should beat the bigram model by > 0.2 nats (got {vl:.2f} vs {bg:.2f})"

check("mlp_lm", _test_mlp_lm, mlp_lm)

---
## Reference solutions
Try the tasks first. Each cell below is collapsed in Colab: double-click it (or click "Show code") to reveal the solution. Running it also checks the reference against the same tests.

In [ ]:
# @title Solution for Task 1: Counting bigram model
def bigram_loss_solution(alpha=0.1):
    C = np.full((V, V), alpha)
    np.add.at(C, (train[:-1], train[1:]), 1)
    P = C / C.sum(1, keepdims=True)
    return -np.mean(np.log(P[val[:-1], val[1:]]))

check("bigram_loss (reference)", _test_bigram_loss, bigram_loss_solution)

In [ ]:
# @title Solution for Task 2: A neural language model with a context of 3
def mlp_lm_solution(ctx=3, d=8, hidden=64, lr=0.5, steps=1500, seed=0):
    r = np.random.default_rng(seed)
    def windows(seq):
        X = np.stack([seq[i:len(seq) - ctx + i] for i in range(ctx)], 1); return X, seq[ctx:]
    Xtr, Ytr = windows(train); Xva, Yva = windows(val)
    E = r.normal(scale=0.5, size=(V, d)); W1 = r.normal(scale=1 / np.sqrt(ctx * d), size=(ctx * d, hidden)); b1 = np.zeros(hidden)
    W2 = r.normal(scale=1 / np.sqrt(hidden), size=(hidden, V)); b2 = np.zeros(V)
    def forward(X):
        x = E[X].reshape(len(X), -1); h = np.tanh(x @ W1 + b1); return x, h, softmax(h @ W2 + b2)
    losses = []
    for _ in range(steps):
        x, h, p = forward(Xtr); n = len(Ytr)
        losses.append(-np.mean(np.log(p[np.arange(n), Ytr])))
        dz = p.copy(); dz[np.arange(n), Ytr] -= 1; dz /= n
        dW2, db2 = h.T @ dz, dz.sum(0); dh = dz @ W2.T * (1 - h ** 2)
        dW1, db1 = x.T @ dh, dh.sum(0); dx = (dh @ W1.T).reshape(n, ctx, d)
        dE = np.zeros_like(E); np.add.at(dE, Xtr, dx)
        for P_, G_ in ((E, dE), (W1, dW1), (b1, db1), (W2, dW2), (b2, db2)): P_ -= lr * G_
    _, _, pv = forward(Xva)
    mlp_lm_solution.params = (E, W1, b1, W2, b2, ctx)
    return losses, -np.mean(np.log(pv[np.arange(len(Yva)), Yva]))

check("mlp_lm (reference)", _test_mlp_lm, mlp_lm_solution)

---
## Experiments
**Sample from it.** Start from "the" and repeatedly sample the next character at temperature 0.8.

In [ ]:
tl, vl = mlp_lm_solution()
E, W1, b1, W2, b2, ctx = mlp_lm_solution.params
print(f"bigram val loss {bigram_loss_solution():.3f} nats   MLP val loss {vl:.3f} nats   (uniform {np.log(V):.3f})")
g = np.random.default_rng(3); s = [stoi[c] for c in 'the']
for _ in range(120):
    h = np.tanh(E[s[-ctx:]].reshape(1, -1) @ W1 + b1); p = softmax((h @ W2 + b2) / 0.8)[0]
    s.append(g.choice(V, p=p))
print(''.join(chars[i] for i in s))

---
## Takeaways
* A language model is a classifier over the next token, trained with cross-entropy on every position.
* Learned embeddings + a hidden layer generalise beyond raw counts and use more context.
* Transformers replace the fixed-context MLP with attention over the whole prefix, but the training objective is the same.